In [1]:
import torch

ckpt_path = "./weigths/knee_leaderboard_state_dict_fastmri.pt"

ckpt = torch.load(ckpt_path, map_location="cpu")

print("Type of object:", type(ckpt))

if isinstance(ckpt, dict):
    print("\nTop-level keys:")
    for k in ckpt.keys():
        print("  ", k)

    # Common case: checkpoint has a 'state_dict'
    if "state_dict" in ckpt and isinstance(ckpt["state_dict"], dict):
        print("\nKeys inside state_dict:")
        for k in ckpt["state_dict"].keys():
            print("  ", k)
else:
    print("\nObject is not a dict, just printing it:")
    print(ckpt)


Type of object: <class 'dict'>

Top-level keys:
   sens_net.norm_unet.unet.down_sample_layers.0.layers.0.weight
   sens_net.norm_unet.unet.down_sample_layers.0.layers.4.weight
   sens_net.norm_unet.unet.down_sample_layers.1.layers.0.weight
   sens_net.norm_unet.unet.down_sample_layers.1.layers.4.weight
   sens_net.norm_unet.unet.down_sample_layers.2.layers.0.weight
   sens_net.norm_unet.unet.down_sample_layers.2.layers.4.weight
   sens_net.norm_unet.unet.down_sample_layers.3.layers.0.weight
   sens_net.norm_unet.unet.down_sample_layers.3.layers.4.weight
   sens_net.norm_unet.unet.conv.layers.0.weight
   sens_net.norm_unet.unet.conv.layers.4.weight
   sens_net.norm_unet.unet.up_conv.0.layers.0.weight
   sens_net.norm_unet.unet.up_conv.0.layers.4.weight
   sens_net.norm_unet.unet.up_conv.1.layers.0.weight
   sens_net.norm_unet.unet.up_conv.1.layers.4.weight
   sens_net.norm_unet.unet.up_conv.2.layers.0.weight
   sens_net.norm_unet.unet.up_conv.2.layers.4.weight
   sens_net.norm_unet.unet

/tmp/ipykernel_332108/1756923564.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location="cpu")


In [ ]:
import torch
import pytorch_lightning as pl  # only for version string (optional)

old_ckpt_path = "./weigths/knee_leaderboard_state_dict_fastmri.pt"          # input .pt
new_ckpt_path = "./weigths/knee_new_state_dict_fastmri.ckpt"    # output .ckpt

orig = torch.load(old_ckpt_path, map_location="cpu")
print("Loaded type:", type(orig))

if not isinstance(orig, dict):
    raise TypeError(f"Expected a dict checkpoint, got: {type(orig)}")

if "state_dict" in orig:
    raise RuntimeError(
        "This checkpoint already has a 'state_dict' key; "
        "you probably don't want to wrap it again."
    )

# ------- build state_dict with 'varnet.' prefix -------
state_dict = {}
for k, v in orig.items():
    new_k = "varnet." + k
    state_dict[new_k] = v

print("Example key mapping:")
for i, (ok, nk) in enumerate(zip(orig.keys(), state_dict.keys())):
    print(f"  {ok}  ->  {nk}")
    if i >= 4:
        break

# ------- build Lightning-style checkpoint -------
new_ckpt = {
    "epoch": 1,
    "global_step": 20000,
    "pytorch-lightning_version": pl.__version__,
    "state_dict": state_dict,

    # ⬇️ IMPORTANT CHANGE HERE:
    # don't give an empty dict, or Trainer will expect ["fit_loop"] etc.
    # either omit 'loops' completely, or set it to None:
    "loops": None,

    "callbacks": {},          # fine to keep empty
    "optimizer_states": [],   # fine to keep empty
    "lr_schedulers": [],
    "hparams_name": None,
    "hyper_parameters": {},   # optional, you can add your hparams if you want
}

torch.save(new_ckpt, new_ckpt_path)
print(f"Saved new checkpoint to: {new_ckpt_path}")



/home/paula/miniconda3/envs/mri_varnet/lib/python3.10/site-packages/lightning_lite/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
/tmp/ipykernel_374820/2997387622.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via th

Loaded type: <class 'dict'>
Example key mapping:
  sens_net.norm_unet.unet.down_sample_layers.0.layers.0.weight  ->  varnet.sens_net.norm_unet.unet.down_sample_layers.0.layers.0.weight
  sens_net.norm_unet.unet.down_sample_layers.0.layers.4.weight  ->  varnet.sens_net.norm_unet.unet.down_sample_layers.0.layers.4.weight
  sens_net.norm_unet.unet.down_sample_layers.1.layers.0.weight  ->  varnet.sens_net.norm_unet.unet.down_sample_layers.1.layers.0.weight
  sens_net.norm_unet.unet.down_sample_layers.1.layers.4.weight  ->  varnet.sens_net.norm_unet.unet.down_sample_layers.1.layers.4.weight
  sens_net.norm_unet.unet.down_sample_layers.2.layers.0.weight  ->  varnet.sens_net.norm_unet.unet.down_sample_layers.2.layers.0.weight
Saved new checkpoint to: ./weigths/knee_new_state_dict_fastmri.ckpt


In [2]:
ckpt_path = "./logs/varnet_msk_mrpro2/epoch=5-step=299886.ckpt"

ckpt = torch.load(ckpt_path, map_location="cpu")

print("Type of object:", type(ckpt))

if isinstance(ckpt, dict):
    print("\nTop-level keys:")
    for k in ckpt.keys():
        print("  ", k)

    # Common case: checkpoint has a 'state_dict'
    if "state_dict" in ckpt and isinstance(ckpt["state_dict"], dict):
        print("\nKeys inside state_dict:")
        for k in ckpt["state_dict"].keys():
            print("  ", k)
else:
    print("\nObject is not a dict, just printing it:")
    print(ckpt)

/tmp/ipykernel_332108/201064004.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(ckpt_path, map_location="cpu")


Type of object: <class 'dict'>

Top-level keys:
   epoch
   global_step
   pytorch-lightning_version
   state_dict
   loops
   callbacks
   optimizer_states
   lr_schedulers
   hparams_name
   hyper_parameters

Keys inside state_dict:
   varnet.sens_net.norm_unet.unet.down_sample_layers.0.layers.0.weight
   varnet.sens_net.norm_unet.unet.down_sample_layers.0.layers.4.weight
   varnet.sens_net.norm_unet.unet.down_sample_layers.1.layers.0.weight
   varnet.sens_net.norm_unet.unet.down_sample_layers.1.layers.4.weight
   varnet.sens_net.norm_unet.unet.down_sample_layers.2.layers.0.weight
   varnet.sens_net.norm_unet.unet.down_sample_layers.2.layers.4.weight
   varnet.sens_net.norm_unet.unet.down_sample_layers.3.layers.0.weight
   varnet.sens_net.norm_unet.unet.down_sample_layers.3.layers.4.weight
   varnet.sens_net.norm_unet.unet.conv.layers.0.weight
   varnet.sens_net.norm_unet.unet.conv.layers.4.weight
   varnet.sens_net.norm_unet.unet.up_conv.0.layers.0.weight
   varnet.sens_net.norm_une